## AR($p$)

This notebook predicts the returns of all tickers with AR($p$) models.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

from utils import extract, WFCV


def feature_engineering_arp(df, p):
    for lag in range(1, p + 1):
        df[f"lag_{lag}"] = df["log_return"].shift(lag)
    df = df.dropna()
    return df


class SimpleOLS:
    def __init__(self, add_intercept=True):
        self.add_intercept = add_intercept
        self.beta = None

    def fit(self, X, y):
        X_mat = X.values if isinstance(X, pd.DataFrame) else X
        if self.add_intercept:
            X_mat = np.column_stack([np.ones(len(X_mat)), X_mat])
        self.beta = np.linalg.pinv(X_mat.T @ X_mat) @ X_mat.T @ y

    def predict(self, X):
        X_mat = X.values if isinstance(X, pd.DataFrame) else X
        if self.add_intercept:
            X_mat = np.column_stack([np.ones(len(X_mat)), X_mat])
        return X_mat @ self.beta


def get_data(path="Datasets/returns_all.csv"):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df


def stats_forecasting_returns_only(name_ticker, model, p=1):
    returns_all = get_data()
    df = extract(returns_all, name_ticker)

    df["log_return"] = np.log1p(df["return"])
    df["log_return"] = df["log_return"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["log_return"])

    df = feature_engineering_arp(df, p=p)
    features = [f"lag_{i}" for i in range(1, p + 1)]
    X = df[features]
    y = df["log_return"]

    y_pred, y_truth, mse_tab, r2 = WFCV(X, y, model)
    reg = sm.OLS(y_truth, sm.add_constant(y_pred)).fit()

    return {
        "SYMBOL": name_ticker,
        "MSE": float(np.mean(mse_tab)) if len(mse_tab) else np.nan,
        "OLS_R2": float(reg.rsquared),
        "OLS_Intercept": float(reg.params[0]),
        "OLS_Slope": float(reg.params[1]),
        "OLS_P_Value_Intercept": float(reg.pvalues[0]),
        "P_Value_Slope": float(reg.pvalues[1]),
    }


df = pd.read_csv('Datasets/returns_all.csv')
df.drop(['KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index'], axis=1, inplace=True)
all_tickers = df.columns[1:].tolist()
tickers_name = "all_tickers"

model = SimpleOLS(add_intercept=True)

results_by_p = {}
for p in [2, 5, 10]:
    results_p = []
    for name in all_tickers:
        results_p.append(stats_forecasting_returns_only(name, model, p=p))
    df_p = pd.DataFrame(results_p)
    results_by_p[p] = df_p

    csv_name = f"Resultats/resultats_{tickers_name}_AR({p})_log1p_returns_only.csv"
    df_p.to_csv(csv_name, index=False)

df_final_ar2 = results_by_p[2]
df_final_ar5 = results_by_p[5]
df_final_ar10 = results_by_p[10]


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p